In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import os
import json
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
import requests
import gc
from datetime import timedelta

# 1. Датасет с предзагрузкой и Word2Vec
class MinecraftHouseDataset(Dataset):
    def __init__(self, data_dir, grid_size=32, global_blocklist_file=None, embedding_dim=16):
        self.data_dir = data_dir
        self.grid_size = grid_size
        self.files = []
        self.data_cache = {}
        self.token_dict = {}
        self.block_keys = []
        self.processed_names = {}
        self.embedding_dim = embedding_dim

        if os.path.exists("word2vec.model"):
            print("Загрузка существующей модели Word2Vec...")
            self.w2v_model = Word2Vec.load("word2vec.model")
            actual_embedding_dim = self.w2v_model.vector_size
            if actual_embedding_dim != embedding_dim:
                print(f"Предупреждение: Размерность эмбеддингов в модели ({actual_embedding_dim}) не совпадает с заданной ({embedding_dim}). Переобучение модели...")
                os.remove("word2vec.model")
                self.w2v_model = None
        if not os.path.exists("word2vec.model"):
            print("Обучение модели Word2Vec...")
            if not global_blocklist_file or not os.path.exists(global_blocklist_file):
                raise ValueError("Файл global_blocklist_file не найден. Требуется для обучения Word2Vec.")
            with open(global_blocklist_file, 'r') as f:
                block_mapping = json.load(f)
            sentences = []
            for block_id, block_str in block_mapping.items():
                if isinstance(block_str, dict):
                    if 'name' in block_str:
                        block_str = block_str['name']
                    else:
                        print(f"Пропущен некорректный блок {block_id}: {block_str} (отсутствует поле 'name')")
                        continue
                if not isinstance(block_str, str):
                    print(f"Пропущен некорректный блок {block_id}: {block_str} (ожидается строка)")
                    continue
                if '[' in block_str and ']' in block_str:
                    try:
                        base_block, states = block_str.split('[', 1)
                        states = states.rstrip(']').split(',')
                        tokens = [base_block] + [str(state) for state in states if isinstance(state, str)]
                    except Exception as e:
                        print(f"Ошибка обработки блока {block_id}: {e}. Пропущен.")
                        continue
                else:
                    tokens = [str(block_str)]
                sentences.append(tokens)
                self.processed_names[block_id] = block_str
            self.w2v_model = Word2Vec(sentences, vector_size=embedding_dim, window=5, min_count=1, workers=4)
            self.w2v_model.save("word2vec.model")
            print("Модель Word2Vec сохранена как word2vec.model")

        has_subfolders = any(os.path.isdir(os.path.join(data_dir, item)) for item in os.listdir(data_dir))
        if has_subfolders:
            print(f"Сканирование подпапок в {data_dir}")
            for folder_name in os.listdir(data_dir):
                folder_path = os.path.join(data_dir, folder_name)
                if os.path.isdir(folder_path):
                    npy_files = [f for f in os.listdir(folder_path) if f.endswith('_normalized.npy')]
                    if len(npy_files) != 1:
                        print(f"В {folder_path} найдено {len(npy_files)} .npy файлов, ожидается 1")
                        continue
                    npy_file = os.path.join(folder_path, npy_files[0])
                    self.files.append(npy_file)
        else:
            print(f"Сканирование файлов в {data_dir} (нет подпапок)")
            npy_files = [f for f in os.listdir(data_dir) if f.endswith('_normalized.npy')]
            if not npy_files:
                raise ValueError(f"В {data_dir} не найдено ни одного .npy файла")
            for npy_file in npy_files:
                self.files.append(os.path.join(data_dir, npy_file))

        if not self.files:
            raise ValueError(f"Не найдено ни одного подходящего .npy файла в {data_dir}")

        if global_blocklist_file and os.path.exists(global_blocklist_file):
            with open(global_blocklist_file, 'r') as f:
                block_mapping = json.load(f)
            self.max_id = max(int(id) for id in block_mapping.keys()) + 2
            self.token_dict = {int(k): (self.processed_names.get(k, str(v) if isinstance(v, str) else v.get('name', str(v))),
                                      self.w2v_model.wv[self.processed_names.get(k, str(v) if isinstance(v, str) else v.get('name', str(v)))])
                             for k, v in block_mapping.items() if self.processed_names.get(k, str(v) if isinstance(v, str) else v.get('name', str(v))) in self.w2v_model.wv}
            self.block_keys = list(self.token_dict.keys())
        else:
            self.max_id = 28000 + 1

        self.data_cache = {}
        for file in tqdm(self.files, desc="Преобразование данных в эмбеддинги"):
            structure = np.load(file).astype(np.int64)
            embedding_structure = np.zeros((grid_size, grid_size, grid_size, self.embedding_dim))
            for x in range(grid_size):
                for y in range(grid_size):
                    for z in range(grid_size):
                        key = structure[x, y, z].item()
                        embedding_structure[x, y, z] = self.token_dict.get(key, (None, np.zeros(self.embedding_dim)))[1]
            self.data_cache[file] = torch.from_numpy(embedding_structure).float()
        air_ratios = []
        for data in self.data_cache.values():
            data_tensor = data.to(torch.float32)
            is_air = torch.all(data_tensor == 0, dim=-1).float()
            air_ratio = torch.mean(is_air).item()
            air_ratios.append(air_ratio)
        self.air_ratio = np.mean(air_ratios)
        print(f"Средняя доля воздуха: {self.air_ratio:.2%}")
        torch.cuda.empty_cache()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        house = self.data_cache[self.files[idx]]
        mask = torch.ones_like(house)
        house = house.permute(3, 0, 1, 2)
        if house.dim() != 4:
            house = house.squeeze(0)
        return house.unsqueeze(0), mask

# 2. VQGAN с дискриминатором
class VQGANFullRes(nn.Module):
    def __init__(self, grid_size=32, num_blocks=28000, codebook_size=512, embedding_dim=16):
        super(VQGANFullRes, self).__init__()
        self.grid_size = grid_size
        self.num_blocks = num_blocks
        self.codebook_size = codebook_size
        self.embedding_dim = embedding_dim

        self.encoder = nn.Sequential(
            nn.Conv3d(embedding_dim, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv3d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv3d(64, embedding_dim, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(embedding_dim),
        )

        self.embedding = nn.Embedding(codebook_size, embedding_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(embedding_dim, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose3d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose3d(32, num_blocks, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

        self.discriminator = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv3d(64, 1, kernel_size=4, stride=1, padding=0),
            nn.Sigmoid(),
        )

    def quantize(self, z):
        z_flattened = z.permute(0, 2, 3, 4, 1).reshape(-1, self.embedding_dim)
        distances = torch.cdist(z_flattened, self.embedding.weight)
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).reshape(z.shape)
        return z_q, encoding_indices

    def forward(self, x):
        x = x.float()
        z = self.encoder(x)
        z_q, indices = self.quantize(z)
        x_recon = self.decoder(z_q)
        return x_recon, indices

    def discriminate(self, x):
        if x.dim() == 5 and x.size(1) == 1:
            return self.discriminator(x)
        raise ValueError(f"Неверная размерность входных данных для дискриминатора: {x.shape}")

# 3. Discrete Diffusion с DDIM
class DiscreteDiffusion(nn.Module):
    def __init__(self, codebook_size=512, timesteps=500, embedding_dim=16):
        super(DiscreteDiffusion, self).__init__()
        self.codebook_size = codebook_size
        self.timesteps = timesteps
        self.embedding_dim = embedding_dim
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=4, batch_first=True),
            num_layers=4
        )
        self.output_layer = nn.Linear(embedding_dim, codebook_size)

    def forward(self, indices, t):
        batch_size, seq_len, _ = indices.shape
        mask = torch.rand((batch_size, seq_len), device=indices.device) < (t / self.timesteps).view(-1, 1)
        noisy_indices = torch.where(mask.unsqueeze(-1), torch.randint(0, self.codebook_size, indices.shape, device=indices.device), indices)
        x = self.transformer(noisy_indices.float())
        pred = self.output_layer(x)
        return pred

    def sample(self, batch_size, seq_len, device, block_weights=None, steps=500):
        indices = torch.randint(0, self.codebook_size, (batch_size, seq_len), device=device)
        for t in reversed(range(steps)):
            t_tensor = torch.full((batch_size,), t, device=device).long()
            with torch.no_grad():
                pred = self.forward(indices.unsqueeze(-1), t_tensor)
                indices = torch.argmax(pred, dim=-1)
        return indices

# 4. Функция обучения
def gradient_penalty(discriminator, real_data, fake_data, device):
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, 1, device=device)
    alpha = alpha.expand_as(real_data)
    interpolates = alpha * real_data + (1 - alpha) * fake_data
    interpolates = interpolates.requires_grad_(True)
    disc_interpolates = discriminator(interpolates)
    gradients = torch.autograd.grad(
        outputs=disc_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(disc_interpolates),
        create_graph=True,
        retain_graph=True
    )[0]
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty

def train_vqgan_and_diffusion(dataset, num_epochs=10, batch_size=1, accumulation_steps=16, train_diffusion_separately=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vqgan = VQGANFullRes(grid_size=32, num_blocks=dataset.max_id, codebook_size=512, embedding_dim=dataset.embedding_dim).to(device)
    diffusion = DiscreteDiffusion(codebook_size=512, timesteps=1000, embedding_dim=dataset.embedding_dim).to(device)
    optimizer_g = torch.optim.Adam(vqgan.parameters(), lr=1e-3)
    optimizer_d = torch.optim.Adam(vqgan.discriminator.parameters(), lr=1e-3)
    optimizer_diff = torch.optim.Adam(diffusion.parameters(), lr=1e-3)
    scheduler_g = StepLR(optimizer_g, step_size=5, gamma=0.1)
    scheduler_d = StepLR(optimizer_d, step_size=5, gamma=0.1)
    scheduler_diff = StepLR(optimizer_diff, step_size=5, gamma=0.1)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    writer = SummaryWriter()

    print(f"Используется устройство: {device}")

    # Обучение VQGAN
    print("Обучение VQGAN...")
    for epoch in range(num_epochs):
        total_loss_g, total_loss_d = 0, 0
        start_time = torch.cuda.Event(enable_timing=True) if device.type == 'cuda' else time.time()
        end_time = torch.cuda.Event(enable_timing=True) if device.type == 'cuda' else time.time()
        start_time.record() if device.type == 'cuda' else None
        for i, (batch_full, mask) in enumerate(tqdm(dataloader, desc=f"Эпоха {epoch+1}/{num_epochs}")):
            batch_full = batch_full.to(device)
            mask = mask.to(device)
            if batch_full.dim() == 6 and batch_full.size(1) == 1:
                batch_full = batch_full.squeeze(1)
            elif batch_full.dim() > 5:
                batch_full = batch_full.squeeze(0)
            optimizer_g.zero_grad(set_to_none=True)
            optimizer_d.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                recon, indices = vqgan(batch_full)
                if recon.shape[1] != batch_full.shape[1]:
                    recon = recon[:, :batch_full.shape[1], :, :, :]
                g_loss = nn.MSELoss()(recon, batch_full)
                weighted_g_loss = g_loss * mask.mean()

                real_out = vqgan.discriminate(batch_full[:, :1, :, :, :].float())
                fake_out = vqgan.discriminate(recon[:, :1, :, :, :].float())
                d_loss = -torch.mean(torch.log(real_out + 1e-8) + torch.log(1 - fake_out + 1e-8))
                gp = gradient_penalty(vqgan.discriminate, batch_full[:, :1, :, :, :], recon[:, :1, :, :, :], device) * 5.0
                weighted_d_loss = (d_loss + gp) * mask.mean()

            g_loss = weighted_g_loss / accumulation_steps
            scaler.scale(g_loss).backward(retain_graph=True)
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer_g)
                scaler.update()
                optimizer_g.zero_grad(set_to_none=True)

            d_loss = weighted_d_loss / accumulation_steps
            scaler.scale(d_loss).backward(retain_graph=True)
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer_d)
                scaler.update()
                optimizer_d.zero_grad(set_to_none=True)

            total_loss_g += g_loss.item() * accumulation_steps
            total_loss_d += d_loss.item() * accumulation_steps

        end_time.record() if device.type == 'cuda' else None
        torch.cuda.synchronize() if device.type == 'cuda' else None
        epoch_time = start_time.elapsed_time(end_time) / 1000 if device.type == 'cuda' else time.time() - start_time
        remaining_time = (num_epochs - (epoch + 1)) * epoch_time
        avg_g_loss = total_loss_g / len(dataloader)
        avg_d_loss = total_loss_d / len(dataloader)
        print(f"Эпоха {epoch+1}/{num_epochs}, G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}, Время: {epoch_time:.2f}с, Осталось: {str(timedelta(seconds=int(remaining_time)))}")
        writer.add_scalar("Loss/VQGAN", avg_g_loss, epoch)
        writer.add_scalar("Loss/Discriminator", avg_d_loss, epoch)
        scheduler_g.step()
        scheduler_d.step()
        torch.cuda.empty_cache()

    # Обучение Diffusion
    if train_diffusion_separately:
        print("Обучение Diffusion...")
        for epoch in range(num_epochs):
            total_loss_diff = 0
            start_time = torch.cuda.Event(enable_timing=True) if device.type == 'cuda' else time.time()
            end_time = torch.cuda.Event(enable_timing=True) if device.type == 'cuda' else time.time()
            start_time.record() if device.type == 'cuda' else None
            for i, (batch_full, mask) in enumerate(tqdm(dataloader, desc=f"Эпоха {epoch+1}/{num_epochs}")):
                batch_full = batch_full.to(device)
                mask = mask.to(device)
                if batch_full.dim() == 6 and batch_full.size(1) == 1:
                    batch_full = batch_full.squeeze(1)
                elif batch_full.dim() > 5:
                    batch_full = batch_full.squeeze(0)
                optimizer_diff.zero_grad(set_to_none=True)

                with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                    recon, indices = vqgan(batch_full)
                    indices = indices.view(batch_full.size(0), -1)  # Сплющиваем до [batch, seq_len]
                    t = torch.randint(0, diffusion.timesteps, (batch_full.size(0),), device=device)
                    diff_loss = nn.MSELoss()(diffusion.forward(indices.unsqueeze(-1), t), indices.float())
                    weighted_diff_loss = diff_loss * mask.mean()

                diff_loss = weighted_diff_loss / accumulation_steps
                scaler.scale(diff_loss).backward(retain_graph=True)
                if (i + 1) % accumulation_steps == 0:
                    scaler.step(optimizer_diff)
                    scaler.update()
                    optimizer_diff.zero_grad(set_to_none=True)

                total_loss_diff += diff_loss.item() * accumulation_steps

            end_time.record() if device.type == 'cuda' else None
            torch.cuda.synchronize() if device.type == 'cuda' else None
            epoch_time = start_time.elapsed_time(end_time) / 1000 if device.type == 'cuda' else time.time() - start_time
            remaining_time = (num_epochs - (epoch + 1)) * epoch_time
            avg_diff_loss = total_loss_diff / len(dataloader)
            print(f"Эпоха {epoch+1}/{num_epochs}, Loss: {avg_diff_loss:.4f}, Время: {epoch_time:.2f}с, Осталось: {str(timedelta(seconds=int(remaining_time)))}")
            writer.add_scalar("Loss/Diffusion", avg_diff_loss, epoch)
            scheduler_diff.step()
            torch.cuda.empty_cache()

    writer.close()
    torch.cuda.empty_cache()
    return vqgan, diffusion

# 5. Генерация с поддержкой тегов
def generate_house(vqgan, diffusion, dataset, tags=None, api_key=None, device="cuda"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    latent_size = 8 * 8 * 8
    batch_size = 1

    block_weights = np.ones(dataset.max_id) * 0.3
    block_weights[list(dataset.token_dict.keys())] = 1.0
    if tags and os.path.exists("block_tags.json"):
        with open("block_tags.json", "r") as f:
            block_tags = json.load(f)
        for tag in tags:
            if tag in block_tags:
                for block_name in block_tags[tag]:
                    if block_name in [v[0] for v in dataset.token_dict.values()]:
                        key = [k for k, v in dataset.token_dict.items() if v[0] == block_name][0]
                        block_weights[key] *= 1.5
    block_weights /= block_weights.sum()
    block_weights = torch.from_numpy(block_weights).float().to(device)

    indices = diffusion.sample(batch_size, latent_size, device, block_weights=block_weights)
    z_q = vqgan.embedding(indices).reshape(batch_size, vqgan.embedding_dim, 8, 8, 8)
    with torch.no_grad(), autocast():
        low_recon = vqgan.decoder(z_q)
        upsampled = F.interpolate(low_recon, size=(32, 32, 32), mode='trilinear', align_corners=False)
        high_recon, _ = vqgan(upsampled)
    pred_ids = torch.argmax(high_recon, dim=1).cpu().numpy().astype(np.int64)

    if np.any(np.isnan(pred_ids)) or np.any(np.isinf(pred_ids)):
        pred_ids = np.nan_to_num(pred_ids, nan=0, posinf=0, neginf=0)

    description = "Описание недоступно (API не настроен)" if not api_key else \
                  requests.post("https://api.x.ai/grok", json={"prompt": f"Опиши дом в стиле Minecraft: {', '.join(set([dataset.token_dict.get(i, ['Unknown'])[0] for i in pred_ids.flatten()]))}.", "api_key": api_key}).json().get("description", "Ошибка API.")
    np.save("generated_house.npy", pred_ids)
    print(f"Сгенерированный дом сохранён в generated_house.npy. Описание: {description}")
    return pred_ids

# 6. Анализ блоков
def analyze_blocks(data_dir):
    block_counts = {}
    total_blocks = 0
    has_subfolders = any(os.path.isdir(os.path.join(data_dir, item)) for item in os.listdir(data_dir))
    if has_subfolders:
        print(f"Сканирование подпапок в {data_dir}")
        for folder_name in os.listdir(data_dir):
            folder_path = os.path.join(data_dir, folder_name)
            if os.path.isdir(folder_path):
                npy_files = [f for f in os.listdir(folder_path) if f.endswith('_normalized.npy')]
                for npy_file in npy_files:
                    file_path = os.path.join(folder_path, npy_file)
                    house = np.load(file_path)
                    unique, counts = np.unique(house, return_counts=True)
                    for block_id, count in zip(unique, counts):
                        block_counts[int(block_id)] = block_counts.get(int(block_id), 0) + int(count)
                    total_blocks += int(house.size)
    else:
        print(f"Сканирование файлов в {data_dir} (нет подпапок)")
        npy_files = [f for f in os.listdir(data_dir) if f.endswith('_normalized.npy')]
        for npy_file in npy_files:
            file_path = os.path.join(data_dir, npy_file)
            house = np.load(file_path)
            unique, counts = np.unique(house, return_counts=True)
            for block_id, count in zip(unique, counts):
                block_counts[int(block_id)] = block_counts.get(int(block_id), 0) + int(count)
            total_blocks += int(house.size)

    blocklist_path = "/content/global_block_list.json"
    block_mapping = {}
    if os.path.exists(blocklist_path):
        with open(blocklist_path, 'r') as f:
            block_mapping = json.load(f)
        print("Путь до основной палитры:", blocklist_path)

    print("\nАнализ используемых блоков:")
    print(f"Общее количество блоков: {total_blocks}")
    for block_id, count in sorted(block_counts.items(), key=lambda x: x[1], reverse=True):
        block_name = block_mapping.get(str(block_id), f"Неизвестный блок (ID: {block_id})")
        percentage = (count / total_blocks) * 100
        print(f"ID: {block_id}, Название: {block_name}, Количество: {count}, Доля: {percentage:.2f}%")

    with open('block_analysis.json', 'w') as f:
        json.dump({str(k): int(v) for k, v in block_counts.items()}, f)
    print("Статистика сохранена в 'block_analysis.json'")
    torch.cuda.empty_cache()

# 7. Основной код
if __name__ == "__main__":
    data_dir = "/content/Data/Output"
    global_blocklist_file = "/content/global_block_list.json"
    batch_size = 2
    num_epochs = 1
    accumulation_steps = 8

    # Анализ блоков
    analyze_blocks(data_dir)
    torch.cuda.empty_cache()

    # Загрузка датасета
    dataset = MinecraftHouseDataset(data_dir, global_blocklist_file=global_blocklist_file)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # Обучение
    vqgan, diffusion = train_vqgan_and_diffusion(dataset, num_epochs=num_epochs, batch_size=batch_size, accumulation_steps=accumulation_steps)
    torch.save(vqgan.state_dict(), "vqgan_full_res_model.pth")
    torch.save(diffusion.state_dict(), "diffusion_model.pth")
    torch.cuda.empty_cache()

    # Генерация
    tags = ["wooden", "stone"]
    api_key = None
    generated_house = generate_house(vqgan, diffusion, dataset, tags=tags, api_key=api_key)
    torch.cuda.empty_cache()